# 7R-D. Present the retained Environment B results

This notebook prepares figures from the retained results for seeds 2101–2105. The numerical values are entered from the result cards and checked by the assertions below before plotting.

No data are generated, no detectors are fitted, no scores are recalculated and no bootstrap intervals are regenerated. This is a reporting notebook.

Seed 2101 remains the primary Environment B world under the documented rule; seeds 2102–2105 provide replication evidence. The fixed-budget detection difference favours M2-B in all five worlds, but criterion-level support is mixed.

The figures cover detection counts, paired intervals, scenario summaries and the comparison with Environment A. Pooled counts are descriptive, not a new hypothesis test. Code labels `M0` and `M2` refer to the corresponding Environment B configurations unless a figure explicitly compares environments.

Environment A retains H1 not supported. Environment B supports H1-B under the documented primary-world rule and its limitations. Neither environment establishes performance in a real financial institution or retailer.

**Outputs:** figures, summary files and an integrity manifest. These reporting checks do not independently reproduce the upstream experiments.

*Documentation copy, 8 November 2026. Notes edited for clarity; code, saved outputs and execution metadata are unchanged. This copy has not been rerun.*


In [1]:
from pathlib import Path
import json, hashlib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.patches import FancyBboxPatch, FancyArrowPatch

OUT = Path('/kaggle/working/retail_7rd_figures') if Path('/kaggle/working').exists() else Path('retail_7rd_figures')
OUT.mkdir(parents=True, exist_ok=True)
print('Output directory:', OUT.resolve())


Output directory: /kaggle/working/retail_7rd_figures


## 1. Load the retained result-card values

The following cell contains the five-world values transcribed from the result cards and assertions on the reported counts and criteria. The next helper saves each figure in the configured output directory.

This is not an automatic import of the five original evaluation packages. Agreement between entered values and assertions supports internal consistency, not independent verification of every source result. Pooled totals remain descriptive.


In [2]:
worlds = pd.DataFrame([{'seed': 2101, 'role': 'Primary confirmatory', 'test_user_days': 87565, 'alerts': 875, 'm0_tp': 48, 'm2_tp': 53, 'm0_recall': 0.26666666666666666, 'm2_recall': 0.29444444444444445, 'critA_point': -0.002838015677747898, 'critA_lo': -0.013536430626763117, 'critA_hi': -0.0003416300502826132, 'critA_pass': True, 'critB_point': 0.02777777777777779, 'critB_lo': 0.005555555555555591, 'critB_hi': 0.050000000000000044, 'critB_pass': True, 'm3_flag': False, 'criteria_met': True, 'delta_tp': 5}, {'seed': 2102, 'role': 'Replication 1', 'test_user_days': 87779, 'alerts': 877, 'm0_tp': 44, 'm2_tp': 51, 'm0_recall': 0.24444444444444444, 'm2_recall': 0.2833333333333333, 'critA_point': -0.0024543659174191485, 'critA_lo': -0.006749203252784953, 'critA_hi': 0.0008753883805608008, 'critA_pass': False, 'critB_point': 0.03888888888888889, 'critB_lo': 0.005555555555555564, 'critB_hi': 0.07777777777777778, 'critB_pass': True, 'm3_flag': False, 'criteria_met': True, 'delta_tp': 7}, {'seed': 2103, 'role': 'Replication 2', 'test_user_days': 87525, 'alerts': 875, 'm0_tp': 49, 'm2_tp': 56, 'm0_recall': 0.2722222222222222, 'm2_recall': 0.3111111111111111, 'critA_point': -0.003194229778464709, 'critA_lo': -0.013005692615147273, 'critA_hi': -0.00018243647463354908, 'critA_pass': True, 'critB_point': 0.03888888888888892, 'critB_lo': 0.005555555555555591, 'critB_hi': 0.07777777777777775, 'critB_pass': True, 'm3_flag': False, 'criteria_met': True, 'delta_tp': 7}, {'seed': 2104, 'role': 'Replication 3', 'test_user_days': 87343, 'alerts': 873, 'm0_tp': 50, 'm2_tp': 53, 'm0_recall': 0.2777777777777778, 'm2_recall': 0.29444444444444445, 'critA_point': -0.0019503688491676512, 'critA_lo': -0.018831094718348755, 'critA_hi': 0.00040321954341023696, 'critA_pass': False, 'critB_point': 0.016666666666666663, 'critB_lo': -0.011111111111111127, 'critB_hi': 0.04444444444444445, 'critB_pass': False, 'm3_flag': False, 'criteria_met': False, 'delta_tp': 3}, {'seed': 2105, 'role': 'Replication 4', 'test_user_days': 87362, 'alerts': 873, 'm0_tp': 45, 'm2_tp': 47, 'm0_recall': 0.25, 'm2_recall': 0.2611111111111111, 'critA_point': -0.0036246014085476366, 'critA_lo': -0.017460703126338885, 'critA_hi': 0.0004628334001485022, 'critA_pass': False, 'critB_point': 0.011111111111111127, 'critB_lo': -0.0111111111111111, 'critB_hi': 0.033333333333333326, 'critB_pass': False, 'm3_flag': False, 'criteria_met': False, 'delta_tp': 2}])
scenarios = pd.DataFrame([{'seed': 2101, 'scenario': 'cross_role_misuse', 'malicious_days': 108, 'm0_detected': 12, 'm2_detected': 17, 'delta_detected': 5}, {'seed': 2101, 'scenario': 'globally_abnormal', 'malicious_days': 36, 'm0_detected': 36, 'm2_detected': 36, 'delta_detected': 0}, {'seed': 2101, 'scenario': 'low_and_slow', 'malicious_days': 36, 'm0_detected': 0, 'm2_detected': 0, 'delta_detected': 0}, {'seed': 2102, 'scenario': 'cross_role_misuse', 'malicious_days': 108, 'm0_detected': 7, 'm2_detected': 15, 'delta_detected': 8}, {'seed': 2102, 'scenario': 'globally_abnormal', 'malicious_days': 36, 'm0_detected': 36, 'm2_detected': 36, 'delta_detected': 0}, {'seed': 2102, 'scenario': 'low_and_slow', 'malicious_days': 36, 'm0_detected': 1, 'm2_detected': 0, 'delta_detected': -1}, {'seed': 2103, 'scenario': 'cross_role_misuse', 'malicious_days': 108, 'm0_detected': 12, 'm2_detected': 19, 'delta_detected': 7}, {'seed': 2103, 'scenario': 'globally_abnormal', 'malicious_days': 36, 'm0_detected': 36, 'm2_detected': 36, 'delta_detected': 0}, {'seed': 2103, 'scenario': 'low_and_slow', 'malicious_days': 36, 'm0_detected': 1, 'm2_detected': 1, 'delta_detected': 0}, {'seed': 2104, 'scenario': 'cross_role_misuse', 'malicious_days': 108, 'm0_detected': 13, 'm2_detected': 17, 'delta_detected': 4}, {'seed': 2104, 'scenario': 'globally_abnormal', 'malicious_days': 36, 'm0_detected': 36, 'm2_detected': 36, 'delta_detected': 0}, {'seed': 2104, 'scenario': 'low_and_slow', 'malicious_days': 36, 'm0_detected': 1, 'm2_detected': 0, 'delta_detected': -1}, {'seed': 2105, 'scenario': 'cross_role_misuse', 'malicious_days': 108, 'm0_detected': 9, 'm2_detected': 11, 'delta_detected': 2}, {'seed': 2105, 'scenario': 'globally_abnormal', 'malicious_days': 36, 'm0_detected': 36, 'm2_detected': 36, 'delta_detected': 0}, {'seed': 2105, 'scenario': 'low_and_slow', 'malicious_days': 36, 'm0_detected': 0, 'm2_detected': 0, 'delta_detected': 0}])
env_a = {'environment': 'A — CERT r6.2', 'test_user_days': 395549, 'test_positive_user_days': 38, 'test_malicious_accounts': 2, 'alerts_each': 3955, 'm0_tp': 5, 'm2_tp': 5, 'm0_recall': 0.13157894736842105, 'm2_recall': 0.13157894736842105, 'criterion_A_point': 0.00108, 'criterion_A_ci95': [-0.00022, 0.00348], 'criterion_A_satisfied': False, 'criterion_B_point': 0.0, 'criterion_B_satisfied': False, 'h1_status': 'NOT SUPPORTED', 'scope': 'Synthetic CERT r6.2 benchmark; not live financial-institution data.'}

# Frozen evidence assertions
assert worlds.seed.tolist() == [2101,2102,2103,2104,2105]
assert worlds.m0_tp.tolist() == [48,44,49,50,45]
assert worlds.m2_tp.tolist() == [53,51,56,53,47]
assert worlds.criteria_met.tolist() == [True,True,True,False,False]
assert not worlds.m3_flag.any()
assert int(worlds.m0_tp.sum()) == 236
assert int(worlds.m2_tp.sum()) == 260
assert int(scenarios.query("scenario == 'cross_role_misuse'").m0_detected.sum()) == 53
assert int(scenarios.query("scenario == 'cross_role_misuse'").m2_detected.sum()) == 79
assert env_a['m0_tp'] == env_a['m2_tp'] == 5
assert env_a['h1_status'] == 'NOT SUPPORTED'

worlds.to_csv(OUT/'final_world_results.csv', index=False)
scenarios.to_csv(OUT/'scenario_results.csv', index=False)
with open(OUT/'environment_a_reference.json','w') as f: json.dump(env_a,f,indent=2)

display(worlds[['seed','role','m0_tp','m2_tp','critA_pass','critB_pass','criteria_met']])


,seed,role,m0_tp,m2_tp,critA_pass,critB_pass,criteria_met
0,2101,Primary confirmatory,48,53,True,True,True
1,2102,Replication 1,44,51,False,True,True
2,2103,Replication 2,49,56,True,True,True
3,2104,Replication 3,50,53,False,False,False
4,2105,Replication 4,45,47,False,False,False


In [3]:
def save_figure(fig, stem):
    paths=[]
    for ext in ('png','pdf','svg'):
        p=OUT/f'{stem}.{ext}'
        fig.savefig(p, dpi=220, bbox_inches='tight')
        paths.append(p)
    plt.close(fig)
    print(stem, '->', ', '.join(p.name for p in paths))
    return paths

def pct(x):
    return 100*x


## Figure 1: Fixed-budget detections in each world

Compare M0-B and M2-B at the common 1.0% budget for seeds 2101–2105. Each world contains 180 malicious user-days.

The bars show detection counts. They do not, on their own, establish whether the paired improvement criterion is met; the interval and criterion figures provide that information.


In [4]:
x=np.arange(len(worlds)); w=0.36
fig, ax = plt.subplots(figsize=(9.5,5.6))
ax.bar(x-w/2, worlds.m0_tp, width=w, label='M0: cyber-log baseline')
ax.bar(x+w/2, worlds.m2_tp, width=w, label='M2: curated context')
ax.set_xticks(x, [str(s) for s in worlds.seed])
ax.set_xlabel('Reserved final seed')
ax.set_ylabel('Malicious user-days detected at 1% alert budget')
ax.set_title('Environment B: fixed-budget detections across final worlds')
ax.legend(frameon=False)
ax.grid(axis='y', alpha=.25)
for i,(a,b) in enumerate(zip(worlds.m0_tp, worlds.m2_tp)):
    ax.text(i-w/2,a+.5,str(a),ha='center',fontsize=9)
    ax.text(i+w/2,b+.5,str(b),ha='center',fontsize=9)
save_figure(fig,'fig_retail_final_world_detections')


fig_retail_final_world_detections -> fig_retail_final_world_detections.png, fig_retail_final_world_detections.pdf, fig_retail_final_world_detections.svg


[PosixPath('/kaggle/working/retail_7rd_figures/fig_retail_final_world_detections.png'),
 PosixPath('/kaggle/working/retail_7rd_figures/fig_retail_final_world_detections.pdf'),
 PosixPath('/kaggle/working/retail_7rd_figures/fig_retail_final_world_detections.svg')]

## Figure 2: Criterion B recall differences

Plot the M2-B-minus-M0-B recall difference with its retained paired 95% interval for each world. The displayed values are percentage-point differences.

Positive values favour M2-B. Criterion B requires the whole interval to be above zero, not just the point estimate.


In [5]:
y=np.arange(len(worlds))
point=100*worlds.critB_point.to_numpy(); lo=100*worlds.critB_lo.to_numpy(); hi=100*worlds.critB_hi.to_numpy()
fig, ax = plt.subplots(figsize=(9.5,5.6))
ax.errorbar(point,y,xerr=np.vstack([point-lo,hi-point]),fmt='o',capsize=4)
ax.axvline(0,linewidth=1)
ax.set_yticks(y,[str(s) for s in worlds.seed])
ax.invert_yaxis()
ax.set_xlabel('M2 − M0 recall difference (percentage points)')
ax.set_ylabel('Reserved final seed')
ax.set_title('Criterion B: fixed-budget recall difference with paired 95% intervals')
ax.grid(axis='x', alpha=.25)
save_figure(fig,'fig_retail_criterion_b_intervals')


fig_retail_criterion_b_intervals -> fig_retail_criterion_b_intervals.png, fig_retail_criterion_b_intervals.pdf, fig_retail_criterion_b_intervals.svg


[PosixPath('/kaggle/working/retail_7rd_figures/fig_retail_criterion_b_intervals.png'),
 PosixPath('/kaggle/working/retail_7rd_figures/fig_retail_criterion_b_intervals.pdf'),
 PosixPath('/kaggle/working/retail_7rd_figures/fig_retail_criterion_b_intervals.svg')]

## Figure 3: Criterion A matched-recall differences

Plot the retained M2-B-minus-M0-B false-positive-rate difference at each world's recall anchor, with its paired 95% interval. The display uses percentage-point differences.

Negative values favour M2-B. Criterion A requires the whole interval to be below zero. These intervals are read from the evidence block, not recalculated here.


In [6]:
y=np.arange(len(worlds))
point=100*worlds.critA_point.to_numpy(); lo=100*worlds.critA_lo.to_numpy(); hi=100*worlds.critA_hi.to_numpy()
fig, ax = plt.subplots(figsize=(9.5,5.6))
ax.errorbar(point,y,xerr=np.vstack([point-lo,hi-point]),fmt='o',capsize=4)
ax.axvline(0,linewidth=1)
ax.set_yticks(y,[str(s) for s in worlds.seed])
ax.invert_yaxis()
ax.set_xlabel('M2 − M0 matched-recall FPR difference (percentage points)')
ax.set_ylabel('Reserved final seed')
ax.set_title('Criterion A: matched-recall FPR difference with paired 95% intervals')
ax.grid(axis='x', alpha=.25)
save_figure(fig,'fig_retail_criterion_a_intervals')


fig_retail_criterion_a_intervals -> fig_retail_criterion_a_intervals.png, fig_retail_criterion_a_intervals.pdf, fig_retail_criterion_a_intervals.svg


[PosixPath('/kaggle/working/retail_7rd_figures/fig_retail_criterion_a_intervals.png'),
 PosixPath('/kaggle/working/retail_7rd_figures/fig_retail_criterion_a_intervals.pdf'),
 PosixPath('/kaggle/working/retail_7rd_figures/fig_retail_criterion_a_intervals.svg')]

## Figure 4: Pooled scenario detections

Show all four configurations across cross-role misuse, globally abnormal behaviour and low-and-slow behaviour. Counts combine the five reserved-final worlds.

The denominators are 540 cross-role malicious days, 180 globally abnormal days and 180 low-and-slow days. The cell checks the entered configuration totals before plotting.

These pooled counts are descriptive. They do not introduce a pooled inferential test or replace the seed-2101 H1-B decision.


In [7]:
# Figure 4 — Descriptive scenario detections
# Pooled across the five reserved-final worlds
# All four configurations included

labels = [
    'Cross-role misuse',
    'Globally abnormal',
    'Low-and-slow'
]

pooled = pd.DataFrame({
    'M0-B': [53, 180, 3],
    'M1-B': [70, 180, 2],
    'M2-B': [79, 180, 1],
    'M3-B': [32, 180, 3]
}, index=labels)

# Check consistency with retained five-world totals
assert pooled['M0-B'].sum() == 236
assert pooled['M1-B'].sum() == 252
assert pooled['M2-B'].sum() == 260
assert pooled['M3-B'].sum() == 215

x = np.arange(len(labels))
width = 0.18

fig, ax = plt.subplots(figsize=(9.5, 5.6))

models = ['M0-B', 'M1-B', 'M2-B', 'M3-B']

for i, model in enumerate(models):
    offset = (i - 1.5) * width

    bars = ax.bar(
        x + offset,
        pooled[model],
        width=width,
        label=model
    )

    ax.bar_label(
        bars,
        padding=3,
        fontsize=9
    )

ax.set_xticks(x)
ax.set_xticklabels(labels)

ax.set_ylabel(
    'Detected malicious user-days across five worlds'
)

ax.set_ylim(0, 205)

ax.legend(
    frameon=False,
    ncol=4,
    loc='upper left'
)

ax.grid(axis='y', alpha=0.25)
ax.set_axisbelow(True)

fig.tight_layout()

save_figure(
    fig,
    'fig_retail_scenario_pooled_detections'
)


fig_retail_scenario_pooled_detections -> fig_retail_scenario_pooled_detections.png, fig_retail_scenario_pooled_detections.pdf, fig_retail_scenario_pooled_detections.svg


[PosixPath('/kaggle/working/retail_7rd_figures/fig_retail_scenario_pooled_detections.png'),
 PosixPath('/kaggle/working/retail_7rd_figures/fig_retail_scenario_pooled_detections.pdf'),
 PosixPath('/kaggle/working/retail_7rd_figures/fig_retail_scenario_pooled_detections.svg')]

## Figure 5: Primary seed-2101 detections

Show the fixed-budget comparison for the designated primary world: M0-B detects 48 of 180 malicious user-days and M2-B detects 53.

The figure's primary-world designation follows the documented supplementary rule. The chart itself does not verify when that rule was fixed.


In [8]:
p=worlds.loc[worlds.seed.eq(2101)].iloc[0]
fig, ax=plt.subplots(figsize=(7.5,5.4))
vals=[p.m0_tp,p.m2_tp]
ax.bar(['M0\nCyber-log baseline','M2\nCurated context'],vals)
ax.set_ylabel('Malicious user-days detected')
ax.set_title('Primary confirmatory world (seed 2101): 1% alert budget')
ax.set_ylim(0,max(vals)+10)
ax.grid(axis='y',alpha=.25)
for i,v in enumerate(vals): ax.text(i,v+.8,f'{int(v)}/180',ha='center')
ax.text(.5,5,'Same alert capacity: 875 alerts per model',ha='center',transform=ax.transAxes)
save_figure(fig,'fig_retail_seed2101_fixed_budget')


fig_retail_seed2101_fixed_budget -> fig_retail_seed2101_fixed_budget.png, fig_retail_seed2101_fixed_budget.pdf, fig_retail_seed2101_fixed_budget.svg


[PosixPath('/kaggle/working/retail_7rd_figures/fig_retail_seed2101_fixed_budget.png'),
 PosixPath('/kaggle/working/retail_7rd_figures/fig_retail_seed2101_fixed_budget.pdf'),
 PosixPath('/kaggle/working/retail_7rd_figures/fig_retail_seed2101_fixed_budget.svg')]

## Figure 6: Primary seed-2101 false positives at matched recall

Compare the false positives required to reach 48 detected malicious days: 741 for M0-B and 493 for M2-B, a difference of 248.

These are matched-recall operating points, not the fixed 875-alert queues. The reduction is a count of false-positive user-days in the simulation, not a measurement of analyst time or cost.


In [9]:
fig, ax=plt.subplots(figsize=(7.5,5.4))
vals=[741,493]
ax.bar(['M0\nCyber-log baseline','M2\nCurated context'],vals)
ax.set_ylabel('False positives at 48 detected malicious days')
ax.set_title('Primary seed 2101: same recall, fewer false positives under M2')
ax.set_ylim(0,850)
ax.grid(axis='y',alpha=.25)
for i,v in enumerate(vals): ax.text(i,v+12,str(v),ha='center')
ax.text(.5,.08,'248 fewer false positives for M2',ha='center',transform=ax.transAxes)
save_figure(fig,'fig_retail_seed2101_matched_recall_fp')


fig_retail_seed2101_matched_recall_fp -> fig_retail_seed2101_matched_recall_fp.png, fig_retail_seed2101_matched_recall_fp.pdf, fig_retail_seed2101_matched_recall_fp.svg


[PosixPath('/kaggle/working/retail_7rd_figures/fig_retail_seed2101_matched_recall_fp.png'),
 PosixPath('/kaggle/working/retail_7rd_figures/fig_retail_seed2101_matched_recall_fp.pdf'),
 PosixPath('/kaggle/working/retail_7rd_figures/fig_retail_seed2101_matched_recall_fp.svg')]

## Figure 7: Criterion outcomes across the five worlds

Display the recorded Criterion A and Criterion B outcomes for the primary world and the four replications. Seeds 2101 and 2103 meet both criteria, seed 2102 meets Criterion B, and seeds 2104 and 2105 meet neither.

The table is a view of the retained comparison flags. Replication outcomes qualify the finding but cannot rescue, replace or redefine the designated primary result.


In [10]:
fig, ax=plt.subplots(figsize=(10.5,5.7))
ax.axis('off')
cols=['Seed','Role','Criterion A','Criterion B','Run-level']
rows=[]
for r in worlds.itertuples(index=False):
    rows.append([str(r.seed),r.role,'PASS' if r.critA_pass else 'Not met','PASS' if r.critB_pass else 'Not met','Criteria met' if r.criteria_met else 'Criteria not met'])
tbl=ax.table(cellText=rows,colLabels=cols,loc='center',cellLoc='center')
tbl.auto_set_font_size(False); tbl.set_fontsize(10); tbl.scale(1,1.7)
ax.set_title('Environment B: predeclared primary and replication worlds',pad=18)
save_figure(fig,'fig_retail_final_world_decision_matrix')


fig_retail_final_world_decision_matrix -> fig_retail_final_world_decision_matrix.png, fig_retail_final_world_decision_matrix.pdf, fig_retail_final_world_decision_matrix.svg


[PosixPath('/kaggle/working/retail_7rd_figures/fig_retail_final_world_decision_matrix.png'),
 PosixPath('/kaggle/working/retail_7rd_figures/fig_retail_final_world_decision_matrix.pdf'),
 PosixPath('/kaggle/working/retail_7rd_figures/fig_retail_final_world_decision_matrix.svg')]

## Figure 8: Compare the two environments

Summarise the Environment A and primary Environment B findings in separate panels. The environments have different populations, prevalence and data-generating structures, so their detection counts are not placed on one shared performance scale.

The contrast supports a bounded discussion of contextual utility. It is not a formal test that the contextual effects differ between environments or proof of the cause of that difference.


In [11]:
fig, ax=plt.subplots(figsize=(11,5.6)); ax.axis('off')
left=FancyBboxPatch((.04,.20),.40,.62,boxstyle='round,pad=0.02',fill=False,linewidth=1.5,transform=ax.transAxes)
right=FancyBboxPatch((.56,.20),.40,.62,boxstyle='round,pad=0.02',fill=False,linewidth=1.5,transform=ax.transAxes)
ax.add_patch(left); ax.add_patch(right)
ax.text(.24,.73,'Environment A — CERT r6.2',ha='center',weight='bold',transform=ax.transAxes,fontsize=13)
ax.text(.24,.59,'M0: 5/38 detected\nM2: 5/38 detected',ha='center',va='center',transform=ax.transAxes,fontsize=11)
ax.text(.24,.40,'Criterion A: not satisfied\nCriterion B: not satisfied\nH1: NOT SUPPORTED',ha='center',va='center',transform=ax.transAxes,fontsize=11)
ax.text(.76,.73,'Environment B — Controlled\nretail simulation',ha='center',weight='bold',transform=ax.transAxes,fontsize=12)
ax.text(.76,.59,'Primary seed 2101\nM0: 48/180 detected\nM2: 53/180 detected',ha='center',va='center',transform=ax.transAxes,fontsize=11)
ax.text(.76,.40,'Criterion A: satisfied\nCriterion B: satisfied\nH1-B: SUPPORTED',ha='center',va='center',transform=ax.transAxes,fontsize=11)
ax.text(.50,.09,'Conclusion: the tested context showed conditional, environment-dependent utility — not universal effectiveness.',ha='center',transform=ax.transAxes,fontsize=11)
ax.set_title('Cross-environment interpretation',pad=12)
save_figure(fig,'fig_cross_environment_conclusion')


fig_cross_environment_conclusion -> fig_cross_environment_conclusion.png, fig_cross_environment_conclusion.pdf, fig_cross_environment_conclusion.svg


[PosixPath('/kaggle/working/retail_7rd_figures/fig_cross_environment_conclusion.png'),
 PosixPath('/kaggle/working/retail_7rd_figures/fig_cross_environment_conclusion.pdf'),
 PosixPath('/kaggle/working/retail_7rd_figures/fig_cross_environment_conclusion.svg')]

## Figure 9: Show the Environment B workflow

Present the four stages in order: 7R-A generates the events, 7R-B constructs the feature representations, 7R-C evaluates one world, and 7R-D reports the retained results.

This is a workflow illustration, not a new experimental stage or a separate validation result.


In [12]:
fig, ax=plt.subplots(figsize=(11.5,4.8)); ax.axis('off')
labels=['7R-A\nControlled retail\nevent generator','7R-B\nM0 / M1 / M2 / M3\nfeature construction','7R-C\nFrozen detector +\nCriteria A/B','7R-D\nFigures and\ncross-environment\nsummary']
xs=[.04,.29,.54,.79]
for x,l in zip(xs,labels):
    box=FancyBboxPatch((x,.34),.17,.34,boxstyle='round,pad=0.015',fill=False,linewidth=1.4,transform=ax.transAxes)
    ax.add_patch(box); ax.text(x+.085,.51,l,ha='center',va='center',transform=ax.transAxes,fontsize=9.5)
for x in [.21,.46,.71]:
    ax.add_patch(FancyArrowPatch((x,.51),(x+.07,.51),arrowstyle='->',mutation_scale=15,transform=ax.transAxes))
ax.text(.50,.18,'Development seed 1101 → generator frozen → final seeds 2101–2105',ha='center',transform=ax.transAxes,fontsize=11)
ax.set_title('Supplementary Environment B implementation and evidence flow')
save_figure(fig,'fig_retail_environment_b_workflow')


fig_retail_environment_b_workflow -> fig_retail_environment_b_workflow.png, fig_retail_environment_b_workflow.pdf, fig_retail_environment_b_workflow.svg


[PosixPath('/kaggle/working/retail_7rd_figures/fig_retail_environment_b_workflow.png'),
 PosixPath('/kaggle/working/retail_7rd_figures/fig_retail_environment_b_workflow.pdf'),
 PosixPath('/kaggle/working/retail_7rd_figures/fig_retail_environment_b_workflow.svg')]

## Figure 10: Explain the cross-role mechanism

Use the fictional payroll employee in the diagram to illustrate how activity can resemble another role's ordinary work. The baseline and contextual views provide different references for that activity.

The diagram explains the simulated mechanism. It is not an observed case from a real employee, and it does not establish that the same mechanism explains every additional detection.


In [13]:
fig, ax=plt.subplots(figsize=(11.5,5.6)); ax.axis('off')
items=[(.04,'Fictional employee\nThabo\nPayroll role'),(.29,"Technical activity\nresembles another\nrole's normal work"),(.54,'M0 sees\nactivity +\npersonal history'),(.79,'M2 additionally sees\nrole / department /\nhours / peer context')]
for x,l in items:
    box=FancyBboxPatch((x,.48),.17,.28,boxstyle='round,pad=0.015',fill=False,linewidth=1.4,transform=ax.transAxes)
    ax.add_patch(box); ax.text(x+.085,.62,l,ha='center',va='center',transform=ax.transAxes,fontsize=9.5)
for x in [.21,.46,.71]: ax.add_patch(FancyArrowPatch((x,.62),(x+.07,.62),arrowstyle='->',mutation_scale=15,transform=ax.transAxes))
ax.text(.50,.30,'Environment B asks whether the organisational context adds discrimination beyond the same technical baseline.',ha='center',transform=ax.transAxes,fontsize=11)
ax.text(.50,.18,'Final evidence: the additional fixed-budget detections were consistently concentrated in cross-role misuse.',ha='center',transform=ax.transAxes,fontsize=11)
ax.set_title('Why organisational context can matter in a structured environment')
save_figure(fig,'fig_retail_cross_role_story_mechanism')


fig_retail_cross_role_story_mechanism -> fig_retail_cross_role_story_mechanism.png, fig_retail_cross_role_story_mechanism.pdf, fig_retail_cross_role_story_mechanism.svg


[PosixPath('/kaggle/working/retail_7rd_figures/fig_retail_cross_role_story_mechanism.png'),
 PosixPath('/kaggle/working/retail_7rd_figures/fig_retail_cross_role_story_mechanism.pdf'),
 PosixPath('/kaggle/working/retail_7rd_figures/fig_retail_cross_role_story_mechanism.svg')]

## 11. Export the summary and figure manifest

Save the cross-environment summary and the integrity records for the generated reporting files. The manifest records the exported assets and their hashes.

These hashes identify the reporting files. They do not replace the upstream five-world evidence packages or establish the timing of the experimental decisions.


In [14]:
summary={
 'environment_A':{'H1':'not supported','M0_tp':5,'M2_tp':5,'positive_days':38},
 'environment_B_primary':{'seed':2101,'H1_B':'supported','M0_tp':48,'M2_tp':53,'positive_days':180,'criterion_A':True,'criterion_B':True},
 'replications':{'criteria_met':int(worlds.loc[worlds.seed.ne(2101),'criteria_met'].sum()),'total':4},
 'descriptive_five_worlds':{'M0_tp':int(worlds.m0_tp.sum()),'M2_tp':int(worlds.m2_tp.sum()),'delta':int((worlds.m2_tp-worlds.m0_tp).sum())},
 'descriptive_cross_role':{'M0':int(scenarios.query("scenario=='cross_role_misuse'").m0_detected.sum()),'M2':int(scenarios.query("scenario=='cross_role_misuse'").m2_detected.sum())},
 'claim_boundary':'Controlled simulation only; no real-retailer or financial-institution validation.'
}
with open(OUT/'figure_evidence_summary.json','w') as f: json.dump(summary,f,indent=2)

rows=[]
for p in sorted(OUT.iterdir()):
    if p.is_file() and p.name!='figure_manifest_sha256.csv':
        h=hashlib.sha256(p.read_bytes()).hexdigest()
        rows.append({'filename':p.name,'size_bytes':p.stat().st_size,'sha256':h})
manifest=pd.DataFrame(rows)
manifest.to_csv(OUT/'figure_manifest_sha256.csv',index=False)
display(manifest)
print(json.dumps(summary,indent=2))


,filename,size_bytes,sha256
0,environment_a_reference.json,554,c7fd2b27d89194d59a147e3a6269e546fda7dd75f7802a...
1,fig_cross_environment_conclusion.pdf,25395,1b20fabcd2bdeecb8b09d023801aa6b5aee3cede1ae9b7...
2,fig_cross_environment_conclusion.png,134660,9381b2a86481e4736a389958404b7abc5e6de8f20f0220...
3,fig_cross_environment_conclusion.svg,62800,1919a6f2ece230774424002527b97e7bdc208580e5464e...
4,fig_retail_criterion_a_intervals.pdf,15871,121a91aa5e067cc0056e4f19c5629e0fc90582e5c7ab7b...
5,fig_retail_criterion_a_intervals.png,73724,93558dc8b38e1b92b03cc4692fd5cc600804c40c712f0e...
6,fig_retail_criterion_a_intervals.svg,42805,ee307bcd1b25f11ce20e03603c6ece5741b7709dbd0bda...
7,fig_retail_criterion_b_intervals.pdf,16492,319b4377da72c2043035546e7eb03f429d0a9bb17126ff...
8,fig_retail_criterion_b_intervals.png,69856,7f6ba21a5b73f9c7dd524698e2dd0f66277716b47925ab...
9,fig_retail_criterion_b_intervals.svg,39216,0cb98cdd6d774b15892c6182d4022baccda676b92029c6...


{
  "environment_A": {
    "H1": "not supported",
    "M0_tp": 5,
    "M2_tp": 5,
    "positive_days": 38
  },
  "environment_B_primary": {
    "seed": 2101,
    "H1_B": "supported",
    "M0_tp": 48,
    "M2_tp": 53,
    "positive_days": 180,
    "criterion_A": true,
    "criterion_B": true
  },
  "replications": {
    "criteria_met": 2,
    "total": 4
  },
  "descriptive_five_worlds": {
    "M0_tp": 236,
    "M2_tp": 260,
    "delta": 24
  },
  "descriptive_cross_role": {
    "M0": 53,
    "M2": 79
  },
  "claim_boundary": "Controlled simulation only; no real-retailer or financial-institution validation."
}


## Interpretation of the retained results

In CERT r6.2, the curated context representation satisfies neither meaningful-improvement criterion, so the original H1 remains not supported.

In the separately specified controlled retail simulation, M2-B satisfies both criteria in the designated primary seed-2101 world. The fixed-budget direction favours M2-B in all five worlds, but only two of the four additional replications meet at least one criterion.

The findings are consistent with conditional utility under the specified simulation. They do not establish universal benefit, identify one cause of the cross-environment difference or validate the approach in a real institution. The primary-world rule, generator assumptions, mixed replication and provenance qualifications remain part of that interpretation.
